# DEMONEXT Observation Sandbox

This notebook is a sandbox for learning how to operate the DEMONEXT science and guide cameras and telescope
using an observation (.obs) file read in using the `ObsFile` class.

## WARNING: READ THIS

**Never, Ever, Run All in this notebook at once**  

This was designed to test individual system operations and document them. Run without care you could
try to drive the telescope places you don't want it to go.

In [3]:
import sys
import os
import time
import math
import glob
import datetime
from datetime import date, timedelta

# Windows Component Object Model (COM) client module.

from win32com.client import Dispatch

# modules we need from anaconda

import numpy as np

# FITS writing and handling

from astropy.io import fits

# astropy units, coordinates, and times

from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.coordinates import TETE
from astropy.time import Time

# we use pathlib for platform-agnostic path handling 

from pathlib import Path

# we use YAML for runtime configuration files

import yaml

# We use logging for runtime logs

import logging

# Import all demonext modules

import demonext
from demonext import config, telescope, pdu, camera, focuser, obsfile

# Boolean state convenience translation dictionaries

OnOff = {True:'On',False:'Off'}
YesNo = {True:'Yes',False:'No'}


## Data-taking session startup

**This assumes you have already started the observatory subsystems using the `StartUp_Shutdown.ipynb` notebook**


In [63]:
# platform-agnostic path definition relative to home

configDir = Path.home() / ".demonext/config"
configFile = "demonext.txt"

cfgFile = str(Path.home() / configDir / configFile)

# Read the runtime configuration file

try:
    cfg = config.Config(cfgFile)
except Exception as exp:
    print(f"ERROR: (Config) - {exp}")

# Start the runtime logger 

logDir = demonext.homePath(cfg.config["directories"]["LogDir"])

logFile = str(Path(logDir) / f"eng{demonext.obsDate()}.txt")

logging.basicConfig(filename=logFile,
                    format="%(asctime)s %(name)s %(levelname)s: %(message)s",
                    filemode="a",
                    level=logging.INFO)

# ID for log entries

logger = logging.getLogger("camNotes")

logger.info("Started the DEMONEXT observation script execution development notebook")

# Instantiate DEMONEXT camera, focuser, and telescope classes

# Camera class 

try:
    cam = camera.Camera(cfgFile)
    print("Camera class started")
except Exception as exp:
    msg = f"Cannot initialize Camera instance: {exp}"
    print(f"ERROR: {msg}")
    logger.exception(msg)

# Telescope class

try:
    tel = telescope.Telescope(cfgFile)
    print("Telescope class started")
except Exception as exp:
    msg = f"Cannot initialize Telescope instance: {exp}"
    print(f"ERROR: {msg}")
    logger.exception(msg)

# Focuser class

try:
    foc = focuser.Focuser(cfgFile)
    print("Focuser class started")
except Exception as exp:
    msg = f"Cannot initialize Focuser instance"
    print(f"ERROR: {msg}")
    logger.exception(msg)


### Connect to DEMONEXT observatory systems

Connect to the follwing services, in order:
 * Telescope controller
 * Focuser
 * Science and guide cameras
 * *(Site services - weather and roof status)*


In [72]:
# connect to the telescope controller

logger.info("Connecting to telescope...")

print("Connecting to the telescope mount...")
try:
    tel.connect()
except Exception as exp:
    print(f"Cannot connect to telescope: {exp}")

if tel.connected:
    print(f"Done: Connected to {tel.telName} successfully")
else:
    print(f"Error: Failed to connect to the telescope controller")
    
# turn off tracking if it is on for some reason

if tel.isTracking():
    tel.tracking("off")

# connect the focuser

print("\nConnecting to the focuser...")
try:
    foc.connect()
except Exception as exp:
    print(f"Cannot connect to the focuser: {exp}")

if foc.connected:
    print(f"Done: Connected to {foc.name}")
else:
    print(f"Error: Failed to connect to PWI3 and the Hedrick focusers")

# connect the camera

print("\nConnecting to the cameras...")
try:
    cam.connect()
except Exception as exp:
    print(f"Cannot connect to the cameras: {exp}")

if cam.connected:
    print(f"Done: Connected to MaxIM DL and the cameras")
else:
    print(f"Error: Failed to connect to MaxIM DL and the cameras")

# share the Telescope class object with the Camera class instance

cam.telescope = tel

# share the Focuser class object with the Camera class instance

cam.focuser = foc

Connecting to the telescope mount...
Done: Connected to DEMONEXT SiTech Servo Controller in the OSU Lab successfully

Connecting to the focuser...
Done: Connected to PlaneWave Focuser (PWI3)

Connecting to the cameras...
Done: Connected to MaxIM DL and the cameras


## Initialize and Home the Telescope

Make sure the telescope starts from a valid zero point.

In [78]:
print(f"\nHoming the telescope...")    
logger.info(f"Homing the telescope...")

t0 = time.time()
try:
    tel.home()
except Exception as exp:
    print(f"ERROR: {exp}")

if tel.isHome():
    telInfo = tel.position()
    print(f"Done: Telescope at Home: Alt={telInfo['Alt']:.5f}d Az={telInfo['Az']:.5f}d,tracking {OnOff[tel.isTracking()]}")
else:
    telInfo = tel.position()
    print(f"Warning: Telescope not at Home: Alt={telInfo['Alt']:.5f}d Az={telInfo['Az']:.5f}d, tracking {OnOff[tel.isTracking()]}")

dt = time.time() - t0
print(f"      Execution time {dt:.2f} seconds")


Homing the telescope...
Done: Telescope at Home: Alt=56.86732d Az=180.17428d,tracking Off
      Execution time: 37.24 seconds


## Take Observations

### Observation Steps
 
 * Load the project and target information into the camera object
 * Point the telescope to the object coordinates
 * *(focus the telescope)*
 * *(setup the guider)*
 * Execute the observing sequences, in order

Take images with the camera and make sure they go where they should. End-to-end test of the basic image 
acquisition process.

### Load the observation file

Load and parse a .obs file with a observation to execute.  .obs files are YAML-format ASCII text
files read using the demonet `ObsFile` class.

If the .obs file is read correctly, print the contents of the obsFile.

The `projInfo` dictionary from the obsFile contains project- and obsFile-specific information we want
in the FITS headers of images taken.  We pass this info along to the `Camera.projInfo` property as
shown in the code below.  The `Camera` class instance also separately stores the object ID, so we pass
the target name from the obsFile.

In [91]:
#obsFile = "OSU_XMDs_M1234.obs"
obsFile = "OSU_XMDs_M2345.obs"

try:
    obs = obsfile.ObsFile(obsFile)
    obs.print()
except Exception as exp:
    print(f"ERROR: loadConfig() - {exp}")

# Pass project and observation info to the Camera for the FITS headers

cam.projInfo = obs.projInfo
cam.objectID(obs.name)


Observation file OSU_XMDs_M2345.obs contents:

project:
  PROJECT: OSU_XMDs
  PI_NAME: Pogge
  PI_INST: OSU
  PI_EMAIL: pogge.1@osu.edu
  TARG_RA: 01:23:45.67
  TARG_DEC: -10:20:30.5
  OBSFILE: OSU_XMDs_M2345.obs
  PRIORITY: 1
  GUIDING: auto

target:
  Name: M2345
  RA2000: 01:23:45.67
  Dec2000: -10:20:30.5
  Priority: 1
  GuideMode: auto

sequence:
  NumObs: 1
  obs1: ['g_SDSS', 10.0, 3]
  obs2: ['r_SDSS', 10.0, 3]
  obs3: ['i_SDSS', 10.0, 3]

constraints:
  Airmass: 1.5

status:
  parsed: Y
  validated: N


### Validate target coordinates and point the telescope

In [94]:
# Load the project and object info

cam.projInfo = obs.projInfo
cam.objectID(obs.name)

# the Telescope class has a built-in RA/Dec validator - slewToRADec() will also check
# but it is a good practice to stop before you send bad coordinates.

if tel.validRADec(obs.appRA,obs.appDec):
    print(f"RA/Dec of target {obs.name} are valid")
else:
    print(f"RA/Dec of target {obs.name} are invalid: {tel.msg}")
    #raise RuntimeError(f"Invalid RA/Dec: {tel.msg}") # good practice, even if slewToRADec() will also gripe


RA/Dec of target M2345 are valid


In [96]:
# Slew the telescope

t0 = time.time()
tel.tracking("on")

print(f"Slewing to {obs.name}, RA={obs.RA} Dec={obs.Dec}...")
try:
    tel.slewToRADec(obs.appRA,obs.appDec)
except Exception as exp:
    print(f"ERROR: {exp}")
    tel.tracking("off")
    
telInfo = tel.position()

raStr = tel.dec2sex(telInfo['RA'])
decStr = tel.dec2sex(telInfo['Dec'])

dt = time.time() - t0
print(f"Done: Telescope at {raStr} {decStr}, tracking {OnOff[tel.isTracking()]}, exec time {dt:.2f} seconds")

Slewing to M2345, RA=01:23:45.67 Dec=-10:20:30.5...
Done: Telescope at 01:25:00.24 -10:12:38.54, tracking On, exec time: 33.84 seconds


### Focus the telescope

coming on-site or we'll make a simulator

### Setup the guider

coming on-site or we'll make a simulator

### Execute the observing sequence

In [99]:
# Execute the observing sequence

t0 = time.time()
for seqNum in range(obs.numObs):
    print(f"Starting observing sequence {seqNum+1} of {obs.numObs}:")
    for iobs,pars in enumerate(obs.obsPars):
        print(f"  Start: {pars[2]}x{pars[1]:.1f} sec {pars[0]} image(s)")
        try:
            cam.science(pars)
            dt = time.time() - t0
            print(f"   Done: Finished {pars[0]} image(s), exec time {dt:.2f} sec, last science image {cam.nextFile}")
        except Exception as exp:
            print(f"Error executing imaging sequence Obs{iobs+1}: {exp}")
            raise RuntimeError(f"Error executing imaging sequence Obs{iobs+1}: {exp}")

# all done
dt = time.time() - t0
print(f"Done: completed {obs.numObs} observation sequence: exec time {dt:.2f} seconds") 

Starting observing sequence 1 of 1:
  Start: 3x10.0 sec g_SDSS image(s)
   Done: Finished g_SDSS image(s), exec time 49.83 sec, last science image D:\Data\20250430\sci20250430.0012.fits
  Start: 3x10.0 sec r_SDSS image(s)
   Done: Finished r_SDSS image(s), exec time 91.16 sec, last science image D:\Data\20250430\sci20250430.0015.fits
  Start: 3x10.0 sec i_SDSS image(s)
   Done: Finished i_SDSS image(s), exec time 132.52 sec, last science image D:\Data\20250430\sci20250430.0018.fits
Done: completed 1 observation sequence: exec time 132.52 seconds


## Stop Here

After this step is where we end experimentation at the notebook cell level and park the telescope
and do other "we're done here" disconnection steps.

You must separately shutdown and power-off observatory systems using an active instance of the
`StartUp_Shutdown.ipynb` notebook.

## End the observation sandbox session

### Park/Stow the telescope

How to gracefully park the telescope before disconnecting it. Powering the telescope off is done
in a different class.

When we are done, disconnect the telescope from the STI app to take it offline but still powered off.

In [104]:
logger.info("Stowing the telescope")

print("Stowing the telescope...")

try:
    tel.park()
except Exception as exp:
    print(f"ERROR: {exp}")

# make sure tracking is off

tel.tracking("off")

if tel.isParked():
    telInfo = tel.position()
    print(f"Done: Telescope is Parked: tracking {OnOff[tel.isTracking()]}")
else:
    telInfo = tel.position()
    print(f"Done: Telescope says not Parked: tracking {OnOff[tel.isTracking()]}")

# disconnect from the STI app and close the ASCOM instance

tel.disconnect()


Stowing the telescope...
Done: Telescope is Parked: tracking Off
